# Weekly Project 5 
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.

## 1. Fitting R1.pcd to R2.pcd

In [1]:
import open3d as o3d
import numpy as np
import copy


# helper function for drawing
# If you want it to be more clear set recolor=True
def draw_registrations(source, target, transformation = None, recolor = False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if(recolor):
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if(transformation is not None):
        source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
source = o3d.io.read_point_cloud("global_registration/r1.pcd")
target = o3d.io.read_point_cloud("global_registration/r2.pcd")

# Used for downsampling.
voxel_size = 0.05

# Show models side by side
draw_registrations(source, target)

In [53]:
####
# Downsample and find features here
####

voxel_size=0.05 

# down sample to help computation
source_sample = source.voxel_down_sample(voxel_size)
target_sample = target.voxel_down_sample(voxel_size)

#estimate centroid
source_sample.estimate_normals()
target_sample.estimate_normals()

# Code

radius_normal=voxel_size * 2
search=o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal,max_nn=40)#search parameters

source_fpfh=o3d.pipelines.registration.compute_fpfh_feature(input=source_sample,search_param=search)
target_fpfh=o3d.pipelines.registration.compute_fpfh_feature(input=target_sample,search_param=search)


In [54]:
####
# Call RANSAC here
####

point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False) #initialize class object


corr_length = 0.9
distance_threshold = voxel_size * 1.5

c0 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(corr_length)
c1 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
c2 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.095)

checker_list = [c0,c1,c2]


In [55]:
ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample, 
    source_fpfh, target_fpfh, 
    True,
    distance_threshold,
    point_to_point,
    checkers = checker_list)

draw_registrations(source, target, ransac_result.transformation, True)

## Fitting car1.ply to car2.ply 

In [79]:
source = o3d.io.read_point_cloud("global_registration/car1.ply")
target = o3d.io.read_point_cloud("global_registration/car2.ply")

# Used for downsampling.
voxel_size = 0.05

# Show models side by side
draw_registrations(source, target)

In [ ]:
####
# Downsample and find features here
####

# down sample to help computation
source_sample = source.voxel_down_sample(voxel_size)
target_sample = target.voxel_down_sample(voxel_size)

#estimate centroid
source_sample.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(
        radius=radius_normal,
        max_nn=30
    )
)

target_sample.estimate_normals(
    o3d.geometry.KDTreeSearchParamHybrid(
        radius=radius_normal,
        max_nn=30
    )
)

o3d.visualization.draw_geometries(
    [source_sample, target_sample],
    point_show_normal=True
)
# Code

radius_normal=voxel_size * 5
search=o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal,max_nn=100)#search parameters

source_fpfh=o3d.pipelines.registration.compute_fpfh_feature(input=source_sample,search_param=search)
target_fpfh=o3d.pipelines.registration.compute_fpfh_feature(input=target_sample,search_param=search)


In [81]:
####
# Call RANSAC here
####

point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False) #initialize class object


corr_length = 0.9
distance_threshold = voxel_size * 1.5

c0 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(corr_length)
c1 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
c2 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.095)

checker_list = [c0,c1,c2]


In [82]:
ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample, 
    source_fpfh, target_fpfh, 
    True,
    distance_threshold,
    point_to_point,
    checkers = checker_list)

draw_registrations(source, target, ransac_result.transformation, True)

### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.